In [1]:
# ============================================================
# MODIS Terra Fire / Thermal Anomalies Downloader
# Region: California
# Tile: row 3, col 3
# Output: 6 CSV files, one time window per year from 2020–2025
# ============================================================

# Install first if needed:
# pip install requests pandas mapbox-vector-tile tqdm

import os
import time
import requests
import pandas as pd
import mapbox_vector_tile

from datetime import date, timedelta
from tqdm import tqdm


# ============================================================
# 1. Settings
# ============================================================

LAYER = "MODIS_Terra_Thermal_Anomalies_All"
TILE_MATRIX_SET = "1km"
TILE_MATRIX = 4

# Your discovered California tile
TILE_ROW = 3
TILE_COL = 3

OUT_DIR = "terra_fire_california_2000_2025_windows"
os.makedirs(OUT_DIR, exist_ok=True)

# Approximate California bounding box
CALIFORNIA_BBOX = {
    "lon_min": -124.5,
    "lon_max": -114.0,
    "lat_min": 32.0,
    "lat_max": 42.2,
}

# One wildfire-season window per year (Aug 1 – Oct 31), 2000 through 2025
TIME_WINDOWS = [
    {
        "name": f"california_fire_{year}_wildfire_season",
        "start": date(year, 8, 1),
        "end": date(year, 10, 31),
    }
    for year in range(2000, 2026)
]


# ============================================================
# 2. Helper functions
# ============================================================

def daterange(start_date, end_date):
    """
    Yield every date from start_date to end_date, inclusive.
    """
    current = start_date
    while current <= end_date:
        yield current
        current += timedelta(days=1)


def point_inside_california_bbox(lat, lon):
    """
    Keep only points inside approximate California bounding box.
    """
    if lat is None or lon is None:
        return False

    return (
        CALIFORNIA_BBOX["lat_min"] <= lat <= CALIFORNIA_BBOX["lat_max"]
        and CALIFORNIA_BBOX["lon_min"] <= lon <= CALIFORNIA_BBOX["lon_max"]
    )


def make_readable(record):
    """
    Convert coded values into readable labels.
    """
    daynight_map = {
        "D": "Daytime Fire",
        "N": "Nighttime Fire",
    }

    satellite_map = {
        "A": "Aqua",
        "T": "Terra",
        "Aqua": "Aqua",
        "Terra": "Terra",
    }

    version_map = {
        "6.1NRT": "Collection 6.1 Near Real-Time processing",
        "6.1": "Collection 6.1 Standard processing",
        "6.0NRT": "Collection 6 Near Real-Time processing",
        "6.0": "Collection 6 Standard processing",
        "6.02": "Collection 6 Standard processing",
        "6.03": "Collection 6 Standard processing",
    }

    record["day_night_flag"] = daynight_map.get(
        record["day_night_flag"],
        record["day_night_flag"]
    )

    record["satellite"] = satellite_map.get(
        record["satellite"],
        record["satellite"]
    )

    record["collection_source"] = version_map.get(
        record["collection_source"],
        record["collection_source"]
    )

    return record


def fetch_tile_for_date(day):
    """
    Fetch one GIBS WMTS vector tile for one date and decode thermal anomaly points.
    """
    time_str = day.strftime("%Y-%m-%d")

    url = (
        f"https://gibs.earthdata.nasa.gov/wmts/epsg4326/best/"
        f"{LAYER}/default/{time_str}/{TILE_MATRIX_SET}/"
        f"{TILE_MATRIX}/{TILE_ROW}/{TILE_COL}.mvt"
    )

    try:
        response = requests.get(url, timeout=30)

        if response.status_code != 200:
            print(f"Request failed: {response.status_code}, date={day}")
            return []

        if len(response.content) == 0:
            return []

        decoded = mapbox_vector_tile.decode(response.content)

    except Exception as e:
        print(f"Failed: date={day}, error={e}")
        return []

    records = []

    for layer_name, layer_content in decoded.items():
        features = layer_content.get("features", [])

        for feature in features:
            p = feature.get("properties", {})

            lat = p.get("LATITUDE")
            lon = p.get("LONGITUDE")

            if not point_inside_california_bbox(lat, lon):
                continue

            record = {
                "latitude": lat,
                "longitude": lon,
                "brightness_temp_channel_21_22_K": p.get("BRIGHTNESS"),
                "brightness_temp_channel_31_K": p.get("BRIGHT_T31"),
                "fire_radiative_power_MW": p.get("FRP"),
                "detection_confidence_percent": p.get("CONFIDENCE"),
                "day_night_flag": p.get("DAYNIGHT"),
                "along_scan_pixel_size_km": p.get("SCAN"),
                "along_track_pixel_size_km": p.get("TRACK"),
                "acquisition_date": p.get("ACQ_DATE"),
                "acquisition_time_UTC": p.get("ACQ_TIME"),
                "satellite": p.get("SATELLITE"),
                "collection_source": p.get("VERSION"),
                "unique_identifier": p.get("UID"),

                # Useful for checking/debugging
                "requested_date": time_str,
                "tile_matrix": TILE_MATRIX,
                "tile_row": TILE_ROW,
                "tile_col": TILE_COL,
            }

            records.append(make_readable(record))

    return records


def download_time_window(window):
    """
    Download all points for one selected time window and return a DataFrame.
    No per-window CSV is written; data is combined and saved once at the end.
    """
    window_name = window["name"]
    start_date = window["start"]
    end_date = window["end"]

    print("\n" + "=" * 70)
    print(f"Downloading window: {window_name}")
    print(f"Date range: {start_date} to {end_date}")
    print("=" * 70)

    window_records = []

    dates = list(daterange(start_date, end_date))

    for day in tqdm(dates, desc=window_name):
        daily_records = fetch_tile_for_date(day)
        window_records.extend(daily_records)

        # Be polite to NASA server
        time.sleep(0.05)

    df = pd.DataFrame(window_records)
    df["window_name"] = window_name

    print(f"Total points in this window: {len(df)}")

    return df


# ============================================================
# 3. Run all yearly windows and combine into a single CSV
# ============================================================

all_window_dfs = []

for window in TIME_WINDOWS:
    df_window = download_time_window(window)
    all_window_dfs.append(df_window)

combined_df = pd.concat(all_window_dfs, ignore_index=True)

combined_path = os.path.join(
    OUT_DIR,
    "california_fire_2000_2025_wildfire_season.csv"
)
combined_df.to_csv(combined_path, index=False)

print("\nSummary by window:")
print(combined_df.groupby("window_name").size().rename("num_points"))

print("\nDone.")
print(f"Total rows across all years: {len(combined_df)}")
print(f"Combined CSV saved to: {combined_path}")


Date range: 2000-08-01 to 2000-10-31


california_fire_2000_wildfire_season:   1%|          | 1/92 [00:00<00:23,  3.80it/s]

Request failed: 404, date=2000-08-01
Request failed: 404, date=2000-08-02


california_fire_2000_wildfire_season:   3%|▎         | 3/92 [00:00<00:18,  4.92it/s]

Request failed: 404, date=2000-08-03
Request failed: 404, date=2000-08-04


...

Total points in this window: 260

Date range: 2024-08-01 to 2024-10-31


california_fire_2024_wildfire_season: 100%|██████████| 92/92 [00:31<00:00,  2.90it/s]


Total points in this window: 732

Date range: 2025-08-01 to 2025-10-31


california_fire_2025_wildfire_season: 100%|██████████| 92/92 [00:28<00:00,  3.21it/s]

Total points in this window: 732

Summary by window:
window_name
california_fire_2001_wildfire_season     205
california_fire_2002_wildfire_season    1332
california_fire_2003_wildfire_season    2826
california_fire_2004_wildfire_season     205
california_fire_2005_wildfire_season     394
california_fire_2006_wildfire_season    1496
california_fire_2007_wildfire_season    3118
california_fire_2008_wildfire_season     335
california_fire_2009_wildfire_season    1527
california_fire_2010_wildfire_season     236
california_fire_2011_wildfire_season     291
california_fire_2012_wildfire_season     312
california_fire_2013_wildfire_season     272
california_fire_2014_wildfire_season     175
california_fire_2015_wildfire_season     236
california_fire_2016_wildfire_season    1715
california_fire_2017_wildfire_season     904
california_fire_2018_wildfire_season     364
california_fire_2019_wildfire_season     470
california_fire_2020_wildfire_season    4245
california_fire_2021_wildfire_seaso